#### AULINOR: Gráfico que mostre a evolução da área em km2, de cada um dos itens em 'descricao'

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from glob import glob

# --- Seção de Configuração (já existente no seu código) ---
# Caminho base dos CSVs
csv_dir = "/Users/lina/Library/CloudStorage/GoogleDrive-faccincarolina@gmail.com/Meu Drive/Qgis/shapefiles/rs_aulinor/mapbiomas/final_csv"

# Ler todos os arquivos CSV
csv_files = sorted(glob(os.path.join(csv_dir, "*.csv")))

# Lista de dataframes com o ano incluído
dfs = []
for f in csv_files:
    ano = int(os.path.basename(f).split("_")[2])  # extrai o ano
    df = pd.read_csv(f)
    df["ano"] = ano
    dfs.append(df)

# Unificar tudo em um dataframe só
df_all = pd.concat(dfs, ignore_index=True)

df_municipios = df_all.copy()

# Novo diretório de salvamento
output_dir = "/Users/lina/Library/CloudStorage/GoogleDrive-faccincarolina@gmail.com/Meu Drive/Qgis/shapefiles/rs_aulinor/mapbiomas/graficos_mudanca_uso_do_solo_v2"

# Criar a pasta se não existir
os.makedirs(output_dir, exist_ok=True)

# Definindo as cores padronizadas
cores_personalizadas = {
    "Campo Alagado e Área Pantanosa": "#519799",
    "Formação Campestre": "#d6bc74",
    "Mosaico de Usos": "#ffefc3",
    "Outras Lavouras Temporárias": "#f54ca9",
    "Outras Áreas não Vegetadas": "#db4d4f",
    "Pastagem": "#edde8e",
    "Praia, Duna e Areal": "#ffa07a",
    "Restinga Arbórea": "#02d659",
    "Restinga Herbácea": "#ad5100",
    "Rio, Lago e Oceano": "#2532e4",
    "Arroz": "#c71585",
    "Formação Florestal": "#1f8d49",
    "Silvicultura": "#7a5900",
    "Soja": "#f5b3c8",
    "Área Urbanizada": "#d4271e"
}

# --- Geração do Gráfico de Evolução da Área por Descrição ---

# 1. Agrupar os dados por ano e 'descricao', somando a 'area_km2'
# Isso te dará a área total de cada tipo de uso para cada ano em toda a região AULINOR
df_evolucao_regional = df_municipios.groupby(['ano', 'descricao'])['area_km2'].sum().unstack(fill_value=0)

# Reordenar as colunas (descrições) para seguir a ordem das cores personalizadas se desejar.
# Isso garante que as legendas e o empilhamento no gráfico sigam uma ordem lógica.
# Mas note que se uma 'descricao' não estiver presente em um ano, ela não aparecerá.
# É melhor garantir que o índice de colunas seja o mesmo do dicionário de cores.
colunas_ordenadas = [d for d in cores_personalizadas.keys() if d in df_evolucao_regional.columns]
df_evolucao_regional = df_evolucao_regional[colunas_ordenadas]

# 2. Criar o gráfico
plt.figure(figsize=(14, 8))

# Plotar como um gráfico de área empilhada
# O `stackplot` é ótimo para mostrar como as proporções mudam ao longo do tempo.
# Se preferir linhas individuais, use `df_evolucao_regional.plot(kind='line', ...)`
plt.stackplot(df_evolucao_regional.index,
              df_evolucao_regional.values.T, # Transpor para que cada linha do DF seja uma camada no stackplot
              labels=df_evolucao_regional.columns,
              colors=[cores_personalizadas[desc] for desc in df_evolucao_regional.columns])


plt.title('Evolução da Área por Uso e Cobertura do Solo na Região AULINOR (1985-2023)', fontsize=16)
plt.xlabel('Ano', fontsize=12)
plt.ylabel('Área (km²)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(title='Uso e Cobertura', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(df_evolucao_regional.index) # Garante que todos os anos sejam exibidos no eixo X
plt.tight_layout() # Ajusta o layout para não cortar elementos


# 3. Salvar o gráfico
nome_arquivo_grafico = os.path.join(output_dir, "evolucao_area_aulino_regional.png")
plt.savefig(nome_arquivo_grafico, dpi=300)
plt.close() # Fecha a figura para liberar memória

print(f"Gráfico de evolução da área por uso e cobertura do solo salvo em: {nome_arquivo_grafico}")

Gráfico de evolução da área por uso e cobertura do solo salvo em: /Users/lina/Library/CloudStorage/GoogleDrive-faccincarolina@gmail.com/Meu Drive/Qgis/shapefiles/rs_aulinor/mapbiomas/graficos_mudanca_uso_do_solo_v2/evolucao_area_aulino_regional.png


#### AULINOR: crescimento de cada item em 'descrição' entre 1985 e 2023 e gerar uma tabela excel com isso

In [2]:
import pandas as pd
import os
from glob import glob

# --- Seção de Configuração (já existente no seu código) ---
# Caminho base dos CSVs
csv_dir = "/Users/lina/Library/CloudStorage/GoogleDrive-faccincarolina@gmail.com/Meu Drive/Qgis/shapefiles/rs_aulinor/mapbiomas/final_csv"

# Ler todos os arquivos CSV
csv_files = sorted(glob(os.path.join(csv_dir, "*.csv")))

# Lista de dataframes com o ano incluído
dfs = []
for f in csv_files:
    ano = int(os.path.basename(f).split("_")[2])  # extrai o ano
    df = pd.read_csv(f)
    df["ano"] = ano
    dfs.append(df)

# Unificar tudo em um dataframe só
df_all = pd.concat(dfs, ignore_index=True)

df_municipios = df_all.copy()

# Novo diretório de salvamento
output_dir = "/Users/lina/Library/CloudStorage/GoogleDrive-faccincarolina@gmail.com/Meu Drive/Qgis/shapefiles/rs_aulinor/mapbiomas/graficos_mudanca_uso_do_solo_v2"

# Criar a pasta se não existir
os.makedirs(output_dir, exist_ok=True)

# --- Cálculo de Crescimento/Decrescimento (1985 vs 2023) ---

# 1. Filtrar os dados para os anos de interesse (1985 e 2023)
df_1985 = df_municipios[df_municipios['ano'] == 1985].copy()
df_2023 = df_municipios[df_municipios['ano'] == 2023].copy()

# 2. Somar a área_km2 para cada 'descricao' para cada ano
area_1985 = df_1985.groupby('descricao')['area_km2'].sum().reset_index()
area_1985.rename(columns={'area_km2': 'area_1985_km2'}, inplace=True)

area_2023 = df_2023.groupby('descricao')['area_km2'].sum().reset_index()
area_2023.rename(columns={'area_km2': 'area_2023_km2'}, inplace=True)

# 3. Combinar os dois DataFrames
# Usar um 'outer merge' para garantir que todas as descrições estejam presentes, mesmo se
# elas só aparecerem em um dos anos. Preencher NaNs com 0.
df_comparacao = pd.merge(area_1985, area_2023, on='descricao', how='outer').fillna(0)

# 4. Calcular a mudança absoluta e percentual
df_comparacao['mudanca_absoluta_km2'] = df_comparacao['area_2023_km2'] - df_comparacao['area_1985_km2']

# Evitar divisão por zero para categorias que tinham 0 em 1985
df_comparacao['mudanca_percentual'] = df_comparacao.apply(
    lambda row: (row['mudanca_absoluta_km2'] / row['area_1985_km2']) * 100 if row['area_1985_km2'] != 0 else (100 if row['mudanca_absoluta_km2'] > 0 else 0),
    axis=1
)

# 5. Reordenar as colunas e formatar para melhor visualização
df_comparacao = df_comparacao[[
    'descricao',
    'area_1985_km2',
    'area_2023_km2',
    'mudanca_absoluta_km2',
    'mudanca_percentual'
]]

# Arredondar os valores para duas casas decimais
df_comparacao['area_1985_km2'] = df_comparacao['area_1985_km2'].round(2)
df_comparacao['area_2023_km2'] = df_comparacao['area_2023_km2'].round(2)
df_comparacao['mudanca_absoluta_km2'] = df_comparacao['mudanca_absoluta_km2'].round(2)
df_comparacao['mudanca_percentual'] = df_comparacao['mudanca_percentual'].round(2)


# 6. Salvar em um arquivo Excel
nome_arquivo_excel = os.path.join(output_dir, "comparacao_uso_solo_1985_2023_AULINOR.xlsx")
df_comparacao.to_excel(nome_arquivo_excel, index=False)

print(f"Tabela de comparação de uso do solo salva em: {nome_arquivo_excel}")
print("\n--- Tabela de Comparação ---")
print(df_comparacao)

Tabela de comparação de uso do solo salva em: /Users/lina/Library/CloudStorage/GoogleDrive-faccincarolina@gmail.com/Meu Drive/Qgis/shapefiles/rs_aulinor/mapbiomas/graficos_mudanca_uso_do_solo_v2/comparacao_uso_solo_1985_2023_AULINOR.xlsx

--- Tabela de Comparação ---
                         descricao  area_1985_km2  area_2023_km2  \
0                            Arroz         145.13         281.47   
1   Campo Alagado e Área Pantanosa         196.18         210.25   
2               Formação Campestre         499.04         466.12   
3               Formação Florestal        1263.94        1201.37   
4                  Mosaico de Usos         634.22         700.40   
5      Outras Lavouras Temporárias         394.75         173.36   
6       Outras Áreas não Vegetadas          95.00          49.92   
7                         Pastagem         334.75         262.51   
8              Praia, Duna e Areal         164.10         104.43   
9                 Restinga Arbórea         171.29   

#### Osório, Imbé e Tramandaí: Gerar um gráfico com as mudanças na cobertura do solo e uma tabela excel com o crescimento de cada item em 'descricao'

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from glob import glob

# --- Seção de Configuração (já existente no seu código) ---
# Caminho base dos CSVs
csv_dir = "/Users/lina/Library/CloudStorage/GoogleDrive-faccincarolina@gmail.com/Meu Drive/Qgis/shapefiles/rs_aulinor/mapbiomas/final_csv"

# Ler todos os arquivos CSV
csv_files = sorted(glob(os.path.join(csv_dir, "*.csv")))

# Lista de dataframes com o ano incluído
dfs = []
for f in csv_files:
    ano = int(os.path.basename(f).split("_")[2])  # extrai o ano
    df = pd.read_csv(f)
    df["ano"] = ano
    dfs.append(df)

# Unificar tudo em um dataframe só
df_all = pd.concat(dfs, ignore_index=True)

df_municipios_full = df_all.copy() # Renomeando para evitar conflito com df_municipios dentro do loop

# Novo diretório de salvamento
output_dir = "/Users/lina/Library/CloudStorage/GoogleDrive-faccincarolina@gmail.com/Meu Drive/Qgis/shapefiles/rs_aulinor/mapbiomas/graficos_mudanca_uso_do_solo_v2"

# Criar a pasta se não existir
os.makedirs(output_dir, exist_ok=True)

# Definindo as cores padronizadas
cores_personalizadas = {
    "Campo Alagado e Área Pantanosa": "#519799",
    "Formação Campestre": "#d6bc74",
    "Mosaico de Usos": "#ffefc3",
    "Outras Lavouras Temporárias": "#f54ca9",
    "Outras Áreas não Vegetadas": "#db4d4f",
    "Pastagem": "#edde8e",
    "Praia, Duna e Areal": "#ffa07a",
    "Restinga Arbórea": "#02d659",
    "Restinga Herbácea": "#ad5100",
    "Rio, Lago e Oceano": "#2532e4",
    "Arroz": "#c71585",
    "Formação Florestal": "#1f8d49",
    "Silvicultura": "#7a5900",
    "Soja": "#f5b3c8",
    "Área Urbanizada": "#d4271e"
}

# --- Municípios para processar ---
municipios_interesse = ["Osório", "Imbé", "Tramandaí"]

for municipio in municipios_interesse:
    print(f"\nProcessando dados para o município: {municipio}")

    # 1. Filtrar o DataFrame para o município atual
    df_municipio_atual = df_municipios_full[df_municipios_full['nm_mun'] == municipio].copy()

    if df_municipio_atual.empty:
        print(f"Não foram encontrados dados para o município de {municipio}. Verifique a grafia.")
        continue # Pula para o próximo município no loop

    # --- Geração do Gráfico Stackplot para o Município Atual ---

    # Agrupar os dados por ano e 'descricao', somando a 'area_km2'
    df_evolucao_municipio = df_municipio_atual.groupby(['ano', 'descricao'])['area_km2'].sum().unstack(fill_value=0)

    # Reordenar as colunas (descrições) para seguir a ordem das cores personalizadas
    colunas_ordenadas = [d for d in cores_personalizadas.keys() if d in df_evolucao_municipio.columns]
    # Se houver colunas no DF que não estão em cores_personalizadas, elas serão ignoradas.
    # Se quiser incluí-las com uma cor padrão, adicione-as aqui.
    df_evolucao_municipio = df_evolucao_municipio[colunas_ordenadas]

    # Criar o gráfico stackplot
    plt.figure(figsize=(14, 8))

    # Plotar como um gráfico de área empilhada
    # Filtrar cores_personalizadas para incluir apenas as descrições presentes no DF atual
    cores_para_plot = [cores_personalizadas[desc] for desc in df_evolucao_municipio.columns]

    plt.stackplot(df_evolucao_municipio.index,
                  df_evolucao_municipio.values.T,
                  labels=df_evolucao_municipio.columns,
                  colors=cores_para_plot)

    plt.title(f'Evolução da Área por Uso e Cobertura do Solo em {municipio} (1985-2023)', fontsize=16)
    plt.xlabel('Ano', fontsize=12)
    plt.ylabel('Área (km²)', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.0)
    plt.legend(title='Uso e Cobertura', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(df_evolucao_municipio.index)
    plt.tight_layout()

    # Salvar o gráfico
    nome_arquivo_grafico = os.path.join(output_dir, f"evolucao_area_{municipio.lower().replace(' ', '_')}_stackplot.png")
    plt.savefig(nome_arquivo_grafico, dpi=300)
    plt.close() # Fecha a figura para liberar memória

    print(f"Gráfico stackplot para {municipio} salvo em: {nome_arquivo_grafico}")



Processando dados para o município: Osório
Gráfico stackplot para Osório salvo em: /Users/lina/Library/CloudStorage/GoogleDrive-faccincarolina@gmail.com/Meu Drive/Qgis/shapefiles/rs_aulinor/mapbiomas/graficos_mudanca_uso_do_solo_v2/evolucao_area_osório_stackplot.png

Processando dados para o município: Imbé
Gráfico stackplot para Imbé salvo em: /Users/lina/Library/CloudStorage/GoogleDrive-faccincarolina@gmail.com/Meu Drive/Qgis/shapefiles/rs_aulinor/mapbiomas/graficos_mudanca_uso_do_solo_v2/evolucao_area_imbé_stackplot.png

Processando dados para o município: Tramandaí
Gráfico stackplot para Tramandaí salvo em: /Users/lina/Library/CloudStorage/GoogleDrive-faccincarolina@gmail.com/Meu Drive/Qgis/shapefiles/rs_aulinor/mapbiomas/graficos_mudanca_uso_do_solo_v2/evolucao_area_tramandaí_stackplot.png


In [ ]:
# --- Cálculo de Crescimento/Decrescimento (1985 vs 2023) para o Município Atual ---

    # Filtrar os dados para os anos de interesse (1985 e 2023) para o município atual
    df_1985 = df_municipio_atual[df_municipio_atual['ano'] == 1985].copy()
    df_2023 = df_municipio_atual[df_municipio_atual['ano'] == 2023].copy()

    # Somar a area_km2 para cada 'descricao' para cada ano
    area_1985_mun = df_1985.groupby('descricao')['area_km2'].sum().reset_index()
    area_1985_mun.rename(columns={'area_km2': 'area_1985_km2'}, inplace=True)

    area_2023_mun = df_2023.groupby('descricao')['area_km2'].sum().reset_index()
    area_2023_mun.rename(columns={'area_km2': 'area_2023_km2'}, inplace=True)

    # Combinar os dois DataFrames
    df_comparacao_mun = pd.merge(area_1985_mun, area_2023_mun, on='descricao', how='outer').fillna(0)

    # Calcular a mudança absoluta e percentual
    df_comparacao_mun['mudanca_absoluta_km2'] = df_comparacao_mun['area_2023_km2'] - df_comparacao_mun['area_1985_km2']

    df_comparacao_mun['mudanca_percentual'] = df_comparacao_mun.apply(
        lambda row: (row['mudanca_absoluta_km2'] / row['area_1985_km2']) * 100 if row['area_1985_km2'] != 0 else (100 if row['mudanca_absoluta_km2'] > 0 else 0),
        axis=1
    )

    # Reordenar as colunas e formatar para melhor visualização
    df_comparacao_mun = df_comparacao_mun[[
        'descricao',
        'area_1985_km2',
        'area_2023_km2',
        'mudanca_absoluta_km2',
        'mudanca_percentual'
    ]]

    # Arredondar os valores para duas casas decimais
    df_comparacao_mun['area_1985_km2'] = df_comparacao_mun['area_1985_km2'].round(2)
    df_comparacao_mun['area_2023_km2'] = df_comparacao_mun['area_2023_km2'].round(2)
    df_comparacao_mun['mudanca_absoluta_km2'] = df_comparacao_mun['mudanca_absoluta_km2'].round(2)
    df_comparacao_mun['mudanca_percentual'] = df_comparacao_mun['mudanca_percentual'].round(2)

    # Salvar em um arquivo Excel
    nome_arquivo_excel = os.path.join(output_dir, f"comparacao_uso_solo_1985_2023_{municipio.lower().replace(' ', '_')}.xlsx")
    df_comparacao_mun.to_excel(nome_arquivo_excel, index=False)

    print(f"Tabela de comparação de uso do solo para {municipio} salva em: {nome_arquivo_excel}")
    print(f"\n--- Tabela de Comparação para {municipio} ---")
    print(df_comparacao_mun)

print("\nProcessamento concluído para todos os municípios.")